# 13 — Pengujian Fungsi Rekomendasi (Dual Direction & Input Buku Baru)

> **Proyek NLP BPJS Kesehatan** | Pengujian fungsi utama:
1. **Book-to-Multiple-Competency** (Arah Utama): Judul + abstrak buku -> multi-kompetensi.
2. **Competency-to-Books** (Arah Tambahan): Satu kompetensi -> Top 10 buku katalog.
3. **Pengujian Buku Baru Unseen**: Menguji model terhadap buku baru berbahasa Indonesia dan Inggris.

## A. Setup dan Pemuatan Model

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'dataset_buku_2000.csv').exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'
MODEL_NAMES = ['bm25', 'tfidf', 'e5', 'bge_m3', 'bge_reranker', 'fasttext']
books = pd.read_csv(ROOT / 'data' / 'dataset_buku_2000.csv', dtype={'book_id': str}).fillna('')
profiles = pd.read_csv(OUT / 'profil_kompetensi_query.csv', dtype={'competency_id': str}).fillna('')
print(f'Katalog: {len(books)} | Kompetensi: {len(profiles)}')

## B. Pengujian Arah 1: Book -> Multiple Competencies (Katalog)

In [ ]:
from src.multi_competency import read_model_scores, rank_competencies

matrices = {m: read_model_scores(OUT, m, books['book_id'], profiles['competency_id']) for m in MODEL_NAMES}
sample_book_id = str(books.iloc[0]['book_id'])
print(f'Buku Uji: {books.iloc[0]["judul"]}')

for name, frame in matrices.items():
    res = rank_competencies(frame, sample_book_id, top_k=5)
    assert res['competency_id'].is_unique
    assert res['skor_asli'].is_monotonic_decreasing
    print(f'{name} -> Top match: {res.iloc[0]["competency_id"]} (skor: {res.iloc[0]["skor_asli"]:.4f})')

## C. Pengujian Arah 2: Competency -> Top Books

In [ ]:
from src.multi_competency import rank_books

sample_cid = 'C023'  # Manajemen Risiko
for name, frame in matrices.items():
    top_books = rank_books(frame, sample_cid, books, top_k=5)
    assert len(top_books) == 5
    assert top_books['judul'].str.strip().str.casefold().is_unique
    print(f'[{name}] Top Book for {sample_cid}: {top_books.iloc[0]["judul"]}')

## D. Pengujian Inferensi Buku Baru (Best Model BGE-M3 & Candidate TF-IDF)

In [ ]:
from src.recommender import BookRecommender

# 1. Model Terbaik Terpilih (BGE-M3 dengan allow_fallback=True untuk pengujian offline)
best_model_path = ROOT / 'models' / 'best_model'
best_rec = BookRecommender.load(best_model_path, allow_fallback=True)
print(f'Best Model Loaded: {best_rec.model_name} (Aktual: {best_rec.actual_model_name}, Fallback: {best_rec.is_fallback_active})')

# Uji Buku Baru Bahasa Indonesia
judul_id = 'Manajemen Risiko Operasional Rumah Sakit dan Jaminan Kesehatan Nasional'
abstrak_id = ('Panduan komprehensif identifikasi risiko klinis, mitigasi kepatuhan BPJS Kesehatan, ' 
              'serta tata kelola audit internal fasilitas pelayanan kesehatan.')
recs_id = best_rec.map_book(judul_id, abstrak_id, top_k=5)
print('>>> Hasil Buku Baru (Bahasa Indonesia):')
display(pd.DataFrame(recs_id)[['competency_id', 'competency_name', 'confidence', 'score', 'model']])

# Uji Buku Baru Bahasa Inggris
judul_en = 'Practical Risk Management for Healthcare Organizations'
abstrak_en = ('A structured guide on operational risk assessment, clinical safety, internal controls, ' 
              'and risk governance for healthcare leaders.')
recs_en = best_rec.map_book(judul_en, abstrak_en, top_k=5)
print('>>> Hasil Buku Baru (Bahasa Inggris):')
display(pd.DataFrame(recs_en)[['competency_id', 'competency_name', 'confidence', 'score', 'model']])


## E. Pengujian Penanganan Kasus Ekstrem dan Validasi Error

In [ ]:
tests = []
for name, frame in matrices.items():
    # Uji ID buku tidak ada
    try:
        rank_competencies(frame, 'BOOK_INVALID_9999')
        inv_b = False
    except KeyError:
        inv_b = True
    # Uji ID kompetensi tidak ada
    try:
        rank_books(frame, 'C_INVALID_9999', books)
        inv_c = False
    except KeyError:
        inv_c = True
    tests.append({'model': name, 'uji_book_id_invalid': inv_b, 'uji_competency_id_invalid': inv_c})

test_report = pd.DataFrame(tests)
display(test_report)
assert test_report[['uji_book_id_invalid', 'uji_competency_id_invalid']].all().all()
test_report.to_csv(OUT / '13_laporan_pengujian_fungsi.csv', index=False)
print('PASS — Seluruh pengujian fungsi dan error handling valid.')

## Kesimpulan Tahap

Fungsi rekomendasi dua arah dan inferensi buku baru unseen telah teruji dengan sukses baik untuk buku bahasa Indonesia maupun bahasa Inggris.